<a href="https://colab.research.google.com/github/nemo10-boop/WUEKM/blob/main/FMNIST_AlexNet_WithNoise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from tqdm import tqdm

# Verify GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
# Download and load training dataset
train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)

# Download and load testing dataset
test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

Train dataset size: 60000
Test dataset size: 10000


Initialize and Adapt AlexNet Architecture

In [ ]:
# Load the architecture of AlexNet
model = models.alexnet(weights=None)

# 1. Modify the first layer to accept 1 channel (grayscale) instead of 3 (RGB)
model.features[0] = nn.Conv2d(1, 64, kernel_size=11, stride=4, padding=2)

# 2. Modify the final linear layer to output 10 classes instead of 1000
model.classifier[6] = nn.Linear(model.classifier[6].in_features, 10)

# Move model to GPU
model = model.to(device)
print(model)

AlexNet(
  (features): Sequential(
    (0): Conv2d(1, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (avgpool): AdaptiveAvgPool2d(output_size=(6, 6))
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=9216, out_features=4096, bias=True)
 

Set Loss Function and Optimizer

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

Define Training and Evaluation Engine Functions

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader, desc="Training", leave=False):
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = (correct / total) * 100
    return epoch_loss, epoch_acc

def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Evaluating", leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = (correct / total) * 100
    return epoch_loss, epoch_acc

Define Custom Gaussian Noise Class

In [ ]:
class AddGaussianNoise(object):
    def __init__(self, mean=0.0, std_255=0.0):
        self.mean = mean
        # Convert standard deviation from 0-255 scale to 0.0-1.0 scale
        self.std = std_255 / 255.0

    def __call__(self, tensor):
        if self.std == 0:
            return tensor
        # Generate noise and add it to the image tensor
        noise = torch.randn(tensor.size()) * self.std + self.mean
        noisy_tensor = tensor + noise
        # Clip values to keep them within valid pixel boundaries [-1, 1] after normalization
        return torch.clamp(noisy_tensor, -1.0, 1.0)

    def __repr__(self):
        return f"{self.__class__.__name__}(mean={self.mean}, std_scaled={self.std})"

Configure Noise Level and Load FMNIST Dataset

In [ ]:
# Set the noise level here: Choose 0, 5, or 10
NOISE_LEVEL = 0

# Base pipeline: Resize -> ToTensor -> Normalize to [-1, 1] -> Add Noise
transform_with_noise = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
    AddGaussianNoise(mean=0.0, std_255=NOISE_LEVEL)
])

# Reload the datasets applying the noisy transformation
train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform_with_noise)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)

test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform_with_noise)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

print(f"Loaded Fashion-MNIST with Gaussian Noise Level (std): {NOISE_LEVEL}")

Loaded Fashion-MNIST with Gaussian Noise Level (std): 0


In [ ]:
num_epochs = 5

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"| Train Loss: {train_loss:.4f} Old Train Acc: {train_acc:.2f}% "
          f"| Test Loss: {test_loss:.4f} Test Acc: {test_acc:.2f}%")

Epoch [1/5] | Train Loss: 0.6443 Old Train Acc: 75.62% | Test Loss: 0.3723 Test Acc: 86.09%


Epoch [2/5] | Train Loss: 0.3545 Old Train Acc: 86.89% | Test Loss: 0.3005 Test Acc: 88.95%


Epoch [3/5] | Train Loss: 0.3094 Old Train Acc: 88.57% | Test Loss: 0.2993 Test Acc: 89.18%


Epoch [4/5] | Train Loss: 0.2876 Old Train Acc: 89.41% | Test Loss: 0.2661 Test Acc: 90.19%


Epoch [5/5] | Train Loss: 0.2663 Old Train Acc: 90.21% | Test Loss: 0.2715 Test Acc: 90.09%


In [ ]:
# Set the noise level here: Choose 0, 5, or 10
NOISE_LEVEL = 5

# Base pipeline: Resize -> ToTensor -> Normalize to [-1, 1] -> Add Noise
transform_with_noise = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
    AddGaussianNoise(mean=0.0, std_255=NOISE_LEVEL)
])

# Reload the datasets applying the noisy transformation
train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform_with_noise)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)

test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform_with_noise)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

print(f"Loaded Fashion-MNIST with Gaussian Noise Level (std): {NOISE_LEVEL}")

Loaded Fashion-MNIST with Gaussian Noise Level (std): 5


In [ ]:
num_epochs = 5

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"| Train Loss: {train_loss:.4f} Old Train Acc: {train_acc:.2f}% "
          f"| Test Loss: {test_loss:.4f} Test Acc: {test_acc:.2f}%")

Epoch [1/5] | Train Loss: 0.6725 Old Train Acc: 74.65% | Test Loss: 0.3830 Test Acc: 86.16%


Epoch [2/5] | Train Loss: 0.3750 Old Train Acc: 86.10% | Test Loss: 0.3484 Test Acc: 86.51%


Epoch [3/5] | Train Loss: 0.3304 Old Train Acc: 87.92% | Test Loss: 0.3072 Test Acc: 88.55%


Epoch [4/5] | Train Loss: 0.2967 Old Train Acc: 89.02% | Test Loss: 0.2839 Test Acc: 89.39%


Epoch [5/5] | Train Loss: 0.2790 Old Train Acc: 89.66% | Test Loss: 0.2909 Test Acc: 89.27%


In [ ]:
# Set the noise level here: Choose 0, 5, or 10
NOISE_LEVEL = 10

# Base pipeline: Resize -> ToTensor -> Normalize to [-1, 1] -> Add Noise
transform_with_noise = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
    AddGaussianNoise(mean=0.0, std_255=NOISE_LEVEL)
])

# Reload the datasets applying the noisy transformation
train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform_with_noise)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)

test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform_with_noise)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

print(f"Loaded Fashion-MNIST with Gaussian Noise Level (std): {NOISE_LEVEL}")

Loaded Fashion-MNIST with Gaussian Noise Level (std): 10


In [ ]:
num_epochs = 5

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"| Train Loss: {train_loss:.4f} Old Train Acc: {train_acc:.2f}% "
          f"| Test Loss: {test_loss:.4f} Test Acc: {test_acc:.2f}%")

Epoch [1/5] | Train Loss: 0.7388 Old Train Acc: 72.24% | Test Loss: 0.4298 Test Acc: 83.78%


Epoch [2/5] | Train Loss: 0.4057 Old Train Acc: 84.95% | Test Loss: 0.3333 Test Acc: 87.28%


Epoch [3/5] | Train Loss: 0.3547 Old Train Acc: 86.84% | Test Loss: 0.3581 Test Acc: 86.68%


Epoch [4/5] | Train Loss: 0.3278 Old Train Acc: 87.91% | Test Loss: 0.3066 Test Acc: 88.00%


Epoch [5/5] | Train Loss: 0.3024 Old Train Acc: 88.84% | Test Loss: 0.2979 Test Acc: 89.41%
